### Analyse dominant driver and teams
### find the dominant driver and constructors over the period of time (Their ability /performance across the season)

In [0]:
%run "../00_common/01_environment_config"

In [0]:
from pyspark.sql import functions as F

In [0]:
## Read Gold Fact

fact_path = f"{s3_root_path}/Gold/Kalashri/fact_session_results"

fact_df = (
    spark.read
    .format("delta")
    .load(fact_path)
)

display(fact_df.limit(10))

In [0]:
## Calculate driver performance per season

driver_performance_df = (
    fact_df
    .groupBy("season", "driver_id")
    .agg(
        F.sum("points").alias("total_points"),
        F.count(F.when(F.col("is_win") == True, 1)).alias("total_wins"),
        F.count(F.when(F.col("is_podium") == True, 1)).alias("total_podiums")
    )
)

display(driver_performance_df)

In [0]:
# Read driver dim

drivers_path = f"{s3_root_path}/Gold/Kalashri/dim_drivers"

drivers_df = (
    spark.read
    .format("delta")
    .load(drivers_path)
)

In [0]:
# Add driver name

driver_performance_final_df = (
    driver_performance_df
    .join(
        drivers_df.select("driver_id", "driver_name"),
        "driver_id",
        "left"
    )
)

display(driver_performance_final_df)

In [0]:
# Calculate constructor performance

constructor_performance_df = (
    fact_df
    .groupBy("season", "constructor_id")
    .agg(
        F.sum("points").alias("total_points"),
        F.count(F.when(F.col("is_win") == True, 1)).alias("total_wins"),
        F.count(F.when(F.col("is_podium") == True, 1)).alias("total_podiums")
    )
)

display(constructor_performance_df)

In [0]:
constructors_path = f"{s3_root_path}/Gold/Kalashri/dim_constructors"

constructors_df = (
    spark.read
    .format("delta")
    .load(constructors_path)
)

In [0]:
constructor_performance_final_df = (
    constructor_performance_df
    .join(
        constructors_df.select(
            "constructor_id",
            "constructor_name"
        ),
        "constructor_id",
        "left"
    )
)

display(constructor_performance_final_df)

In [0]:
# driver historical performance

driver_history_df = (
    driver_performance_final_df
    .orderBy("driver_id", "season")
)

display(driver_history_df)

In [0]:
# Constructor historical performance

constructor_history_df = (
    constructor_performance_final_df
    .orderBy("constructor_id", "season")
)

display(constructor_history_df)

In [0]:
# overall driver performance

driver_overall_df = (
    driver_performance_final_df
    .groupBy("driver_id", "driver_name")
    .agg(
        F.sum("total_points").alias("overall_points"),
        F.sum("total_wins").alias("overall_wins"),
        F.sum("total_podiums").alias("overall_podiums")
    )
    .orderBy(F.desc("overall_points"))
)

display(driver_overall_df)

In [0]:
# Overall contructor Performance

constructor_overall_df = (
    constructor_performance_final_df
    .groupBy("constructor_id", "constructor_name")
    .agg(
        F.sum("total_points").alias("overall_points"),
        F.sum("total_wins").alias("overall_wins"),
        F.sum("total_podiums").alias("overall_podiums")
    )
    .orderBy(F.desc("overall_points"))
)

display(constructor_overall_df)

In [0]:
# Save driver performance

driver_target_path = f"{s3_root_path}/Gold/Kalashri/driver_performance"

(
    driver_performance_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(driver_target_path)
)

In [0]:
# Save contructor performance

constructor_target_path = f"{s3_root_path}/Gold/Kalashri/constructor_performance"

(
    constructor_performance_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(constructor_target_path)
)